---

In [1]:
#!/usr/bin/env python3
"""
CMC Clinical Data Conversion to Maastro Format
================================================

LOGIC EXPLANATION:
==================

The model's parse_data.py expects values in UPPERCASE for site/stage/hpv
because it does: str(value).upper().strip() before matching.

But looking at Maastro format (lowercase) vs what parse_data.py accepts,
the code handles both because it uppercases everything internally.

What ACTUALLY matters is the VALUE CONTENT matches the mapping in parse_data.py:

1. SITE:      CMC uses 'Larynx', 'NPX' -> need 'LARYNX', 'NASOPHARYNX'
2. T-STAGE:   CMC uses 'T4a', 'T1b'   -> need 'T4A', 'T1' (model only knows T1/T2/T3/T4/TX)
3. N-STAGE:   CMC uses 'N2a', 'No'    -> need 'N2A', 'N0'
4. M-STAGE:   CMC uses 'Mo', 'M1'     -> need '0', '1' (Maastro style)
5. GROUPSTAGE:CMC uses 'IVA', 'I'     -> need 'IVA', 'I' (already correct, just uppercase)
6. HPV:       CMC uses 'Yes/No/Not available' -> need 'POSITIVE'/'NEGATIVE'/'N/A'

KEY INSIGHT from parse_data.py:
- Model only uses N-stage, T-stage, Volume, Area as features (not site/hpv/groupstage)
- So site/hpv must match for loading but won't affect model training directly
- T4a and T4b both map to 't4' bin - so safe to simplify
- N2a, N2b, N2c all map to 'n2' bin - so safe to simplify
"""

import pandas as pd
import numpy as np

# Load CMC data
df = pd.read_csv('cmc.csv')
print(f"Loaded {len(df)} patients")
print(f"Columns: {list(df.columns)}\n")

# ============================================================================
# FIX 1: Convert tstage/nstage/mstage from float to int string
# Pandas reads columns as 3.0, 1.0 (float) when NaN values are present
# Model expects '3', '1' (string) - '3.0' will NOT match the mapping
# This must be done BEFORE the mapping below
# ============================================================================
df['tstage'] = df['tstage'].apply(lambda x: str(int(x)) if pd.notna(x) else np.nan)
df['nstage'] = df['nstage'].apply(lambda x: str(int(x)) if pd.notna(x) else np.nan)
df['mstage'] = df['mstage'].apply(lambda x: str(int(x)) if pd.notna(x) else np.nan)
print("Fixed float values: 3.0 -> '3', 1.0 -> '1'\n")

# ============================================================================
# FIX 2: Impute 2 patients with missing staging (mode-based + clinical reasoning)
#
# 077630P - Nasopharynx, Stage IVB:
#   Only other Nasopharynx IVB patient (009644P) has T1, N1, M1
#   Most common Nasopharynx T-stage in cohort: T1 (4 out of 6 patients)
#   Most common Nasopharynx N-stage in cohort: N1 (4 out of 6 patients)
#   Stage IVB by AJCC = M1 (distant metastasis present)
#   Imputed: T=1, N=1, M=1
#
# 175799P - Larynx, Stage completely unknown:
#   Most common Larynx T-stage in cohort: T3 (34 out of 54 = 63%)
#   Most common Larynx N-stage in cohort: N0 (40 out of 54 = 74%)
#   T3N0M0 corresponds to Stage III
#   Imputed: T=3, N=0, M=0, groupstage=iii
# ============================================================================
df.loc[df['id'] == '077630P', 'tstage'] = '1'
df.loc[df['id'] == '077630P', 'nstage'] = '1'
df.loc[df['id'] == '077630P', 'mstage'] = '1'

df.loc[df['id'] == '175799P', 'tstage'] = '3'
df.loc[df['id'] == '175799P', 'nstage'] = '0'
df.loc[df['id'] == '175799P', 'mstage'] = '0'
df.loc[df['id'] == '175799P', 'groupstage'] = 'iii'

print("Imputed 077630P: T=1, N=1, M=1 (Nasopharynx Stage IVB, mode-based)")
print("Imputed 175799P: T=3, N=0, M=0, groupstage=iii (Larynx, mode-based)\n")

# ============================================================================
# CONVERSION 1: SITE
# CMC: 'Larynx', 'Hypopharynx', 'Oropharynx', 'NPX'
# Maastro: 'larynx', 'oropharynx'
# parse_data.py expects: 'HYPOPHARYNX', 'LARYNX', 'NASOPHARYNX', 'OROPHARYNX'
# ============================================================================
site_mapping = {
    'Larynx':       'larynx',
    'Hypopharynx':  'hypopharynx',
    'Oropharynx':   'oropharynx',
    'NPX':          'nasopharynx',   # NPX = Nasopharynx
}
df['site'] = df['site'].map(site_mapping).fillna(df['site'])  # keep already-converted values
print("Site conversion:")
print(f"  NPX -> nasopharynx (Nasopharynx)")
print(f"  Others: lowercase to match Maastro\n")

# ============================================================================
# CONVERSION 2: T-STAGE
# CMC: 'T1', 'T1b', 'T2', 'T3', 'T4a', 'T4b'
# Maastro: '1', '2', '3', '4' (just numbers)
# parse_data.py accepts: 'T1','1' / 'T2','2' / 'T3','3' / 'T4','T4A','T4B','4' / 'TX'
# LOGIC: T1b->1, T4a->4, T4b->4 (model bins T4A and T4B together as 't4')
# ============================================================================
tstage_mapping = {
    '1':    '1',
    '2':    '2',
    '3':    '3',
    '4':    '4',
    'T1':   '1',
    'T1b':  '1',    # T1b -> T1 bin (model has no T1b category)
    'T2':   '2',
    'T3':   '3',
    'T4':   '4',
    'T4a':  '4',    # T4a -> T4 bin
    'T4b':  '4',    # T4b -> T4 bin
    'TX':   'TX',
}
df['tstage'] = df['tstage'].map(tstage_mapping)
print("T-stage conversion:")
print(f"  T1b -> 1  (no T1b in model, maps to t1 bin)")
print(f"  T4a -> 4  (model bins T4A as t4)")
print(f"  T4b -> 4  (model bins T4B as t4)\n")

# ============================================================================
# CONVERSION 3: N-STAGE
# CMC: 'N1', 'N2', 'N2a', 'N2b', 'N2c', 'N3', 'No'
# Maastro: '0', '1', '2', '3'
# parse_data.py accepts: 'N0','0' / 'N1','1' / 'N2','N2A','N2B','N2C','2' / 'N3','N3B','3'
# LOGIC: 'No' is CMC's way of writing N0. N2a/b/c all go to n2 bin.
# ============================================================================
nstage_mapping = {
    '0':    '0',
    '1':    '1',
    '2':    '2',
    '3':    '3',
    'No':   '0',    # CMC writes 'No' for N0
    'N0':   '0',
    'N1':   '1',
    'N2':   '2',
    'N2a':  '2',    # N2a -> N2 bin
    'N2b':  '2',    # N2b -> N2 bin
    'N2c':  '2',    # N2c -> N2 bin
    'N3':   '3',
    'N3b':  '3',
}
df['nstage'] = df['nstage'].map(nstage_mapping)
print("N-stage conversion:")
print(f"  'No' -> 0  (CMC uses 'No' instead of 'N0')")
print(f"  N2a, N2b, N2c -> 2  (all map to n2 bin in model)\n")

# ============================================================================
# CONVERSION 4: M-STAGE
# CMC: 'Mo', 'M1' or already numeric '0','1' after FIX 1
# Maastro: '0', '1'
# LOGIC: CMC uses 'Mo' (letter o) instead of 'M0' (zero)
# ============================================================================
mstage_mapping = {
    '0':    '0',
    '1':    '1',
    'Mo':   '0',    # CMC uses letter 'o' not zero '0'
    'M0':   '0',
    'M1':   '1',
}
df['mstage'] = df['mstage'].map(mstage_mapping)
print("M-stage conversion:")
print(f"  'Mo' -> 0  (CMC uses letter 'o', Maastro uses digit '0')\n")

# ============================================================================
# CONVERSION 5: GROUP STAGE
# CMC: 'I', 'II', 'III', 'IVA', 'IVB'
# Maastro: 'i', 'ii', 'iii', 'iva', 'ivb', 'ivc'
# parse_data.py accepts: 'STAGE I','I' / 'STAGE II','II' etc.
# LOGIC: Just lowercase to match Maastro format
# ============================================================================
df['groupstage'] = df['groupstage'].str.lower()
print("Group stage conversion:")
print(f"  Lowercase to match Maastro format (I->i, IVA->iva)\n")

# ============================================================================
# CONVERSION 6: HPV STATUS
# CMC: 'Yes', 'No', 'Not available or tested'
# Maastro: 'positive', 'negative'
# parse_data.py accepts: '+','POSITIVE' / '-','NEGATIVE' / 'N/A','NAN','NOT TESTED'
# ============================================================================
hpv_mapping = {
    'Yes':                      'positive',
    'No':                       'negative',
    'Not available or tested':  'N/A',
}
df['overall_hpv_status'] = df['overall_hpv_status'].map(hpv_mapping).fillna(df['overall_hpv_status'])
print("HPV status conversion:")
print(f"  'Yes' -> positive")
print(f"  'No'  -> negative")
print(f"  'Not available or tested' -> N/A\n")

# ============================================================================
# KEEP REQUIRED COLUMNS ONLY (matching Maastro structure)
# ============================================================================
output_columns = [
    'id',
    'site',
    'age',
    'sex',
    'overall_hpv_status',
    'tstage',
    'nstage',
    'mstage',
    'groupstage',
    'event_locoregional_recurrence',
    'locoregional_recurrence_in_days',
    'vol',
    'area',
]

# Keep only columns that exist
output_columns = [c for c in output_columns if c in df.columns]
df_out = df[output_columns].copy()

# Lowercase sex to match Maastro
if 'sex' in df_out.columns:
    df_out['sex'] = df_out['sex'].str.lower()

# ============================================================================
# VALIDATION: Check nothing broke
# ============================================================================
print("="*60)
print("VALIDATION - Final unique values:")
print("="*60)
for col in ['site', 'tstage', 'nstage', 'mstage', 'groupstage', 'overall_hpv_status']:
    if col in df_out.columns:
        nulls = df_out[col].isna().sum()
        print(f"{col}: {sorted(df_out[col].dropna().unique().tolist())} | Missing: {nulls}")

print(f"\nTotal patients: {len(df_out)}")
print(f"LRR events: {df_out['event_locoregional_recurrence'].sum()}")
print(f"Volume missing: {df_out['vol'].isna().sum()}")
print(f"Area missing: {df_out['area'].isna().sum()}")

# Warn if critical columns have missing values
for col in ['tstage', 'nstage']:
    n = df_out[col].isna().sum()
    if n > 0:
        print(f"\nWARNING: {n} missing values in {col}:")
        print(df_out[df_out[col].isna()][['id', col]].to_string())
    else:
        print(f"  No missing values in {col}")

# ============================================================================
# SAVE
# ============================================================================
output_file = 'cmc_converted.csv'
df_out.to_csv(output_file, index=False)
print(f"\nSaved: {output_file}")
print(f"  Ready for Mateus et al. model!")

Loaded 102 patients
Columns: ['id', 'Year of Birth', 'sex', 'Smoking History', 'Chewable tobacco', 'Exam Date', 'Weight', 'site', 'tstage', 'nstage', 'mstage', 'groupstage', 'overall_hpv_status', 'Date - Start', 'Date - Finish', 'Start Date', 'Stop Date', 'Date.1', 'Last F/U at CMC   Status at last follow up', 'Date of Documented Recurrence', 'Recurrence.1', 'Local Recurrence', 'Date.2', 'Regional Recurrence', 'Date.3', 'Distant Recurrence', 'Date.4', 'Alive', 'event_locoregional_recurrence', 'LRR_Date', 'months_diff', 'months_diff_lrr', 'locoregional_recurrence_in_days', 'LRR_event', 'vol', 'area']



ValueError: invalid literal for int() with base 10: 'T3'

---